# 01 - BBOB Problem Noise Landscape Analysis

### 🔬 Central Research Question:
> **"How does stochastic noise injection distort underlying fitness landscape topology, gradient signals, and global minimum accessibility across continuous optimization benchmarks?"**

This notebook provides a **rigorous mathematical and empirical analysis of noise injection models** across representative BBOB continuous benchmark functions:
- **$ (Sphere):** Separable unimodal bowl.
- **$ (Rosenbrock):** Ill-conditioned curved valley.
- **{11}$ (Discus):** High conditioning single sensitive direction.
- **{15}$ (Rastrigin Multi-Modal):** Highly multimodal grid with strong global structure.
- **{21}$ (Gallagher 101 Peaks):** 101 asymmetric Gaussian peaks with \text{–}5$ unit local elevation gaps.

---

### 📐 Mathematical Formulations of Compared Noise Models:
1. **Deterministic Clean Ground Truth ($):**
   44409\tilde{f}(x) = f(x)44409

2. **Homoscedastic Additive Noise ($):**
   44409\tilde{f}(x) = f(x) + \mathcal{N}\left(0, (\sigma \cdot \bar{\Delta y})^2\right)44409
   where $\bar{\Delta y} = \mathbb{E}[|f(x) - f^*|]$ is the calibrated mean landscape scale across bounds 5^D$.
   *Key Property:* Constant variance throughout the search domain, creating non-zero evaluation variance directly at the global optimum ^*$.

3. **Heteroscedastic Optimality-Gap Noise ($):**
   44409\tilde{f}(x) = f(x) + \mathcal{N}\left(0, (\sigma \cdot |f(x) - f^*|)^2\right)44409
   *Key Property:* Noise variance scales with the optimality gap $|f(x) - f^*|$. Far from the target, variance challenges exploration; as  \to x^*$, $|f(x) - f^*| \to 0$, preserving the deterministic basin geometry at the global minimum.


In [6]:
import os
import sys
from pathlib import Path

cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import rankdata

from shared.config import DATA_DIR, RESULTS_DIR, PROJECT_ROOT
from evolution.domain.enums import NoiseModelEnum
from evolution.domain.services.noise_strategy import (
    BaseNoiseStrategy,
    NoNoiseStrategy,
    HomoscedasticAdditiveNoiseStrategy,
    HeteroscedasticNoiseStrategy,
    AWGNStrategy,
    NoiseStrategyFactory,
)
from evolution.infra.problems.bbob import BBOBProblem
from benchmarking.domain.enums import BBOBFunction

pub_dir = RESULTS_DIR / "publication" / "noise_landscapes"
pub_dir.mkdir(parents=True, exist_ok=True)

print("✅ Environment initialized with BBOB problem suites and domain noise strategy services.")


✅ Environment initialized with BBOB problem suites and domain noise strategy services.


## 1. Unified Experiment Configuration
Define test problems, dimensions, and noise levels. Set `TARGET_PROBLEM_IDS` to generate figures across all benchmark classes.

In [7]:
TARGET_PROBLEM_IDS = [1, 8, 11, 15, 21]  # All 5 core benchmark problems
dim = 2
instance_id = 1
primary_noise_std = 0.05  # Benchmark standard σ = 0.05
noisy_levels = [0.05, 0.1, 0.2]
all_levels = [0.0] + noisy_levels

# Global aesthetic color palette
color_palette = {
    0.0:  "#0F172A",  # slate dark (clean)
    0.05: "#10B981",  # emerald green
    0.1:  "#06B6D4",  # cyan
    0.2:  "#3B82F6",  # blue
    0.5:  "#8B5CF6",  # purple
    "homo":   "#EF4444", # red (homoscedastic additive noise)
    "hetero": "#10B981"  # emerald (heteroscedastic optimality-gap noise)
}

print(f"🎯 Target Problems Configured ({len(TARGET_PROBLEM_IDS)}): {[f"f{p}: {BBOBFunction.get_name(p)} ({BBOBFunction.get_class(p)})" for p in TARGET_PROBLEM_IDS]}")
print(f"⚡ Primary Noise Level for 3-Panel Demonstrations: σ = {primary_noise_std}")


🎯 Target Problems Configured (5): ['f1: Sphere (f1) (Separable)', 'f8: Rosenbrock (f8) (Low Conditioning)', 'f11: Discus (f11) (High Conditioning)', 'f15: Rastrigin Multi-Modal (f15) (Multi-Modal (Global))', 'f21: Gallagher 101 Peaks (f21) (Multi-Modal (Weak))']
⚡ Primary Noise Level for 3-Panel Demonstrations: σ = 0.05


## 2. 1D Cross-Section 3-Panel Noise Comparison Across Functions

### 🎯 Why we do this plot:
Instead of abstract equations, this **3-Panel Methodology Comparison** directly visualizes continuous 1D cross-sections passing through the global optimum of each BBOB function under different noise regimes.

### 📖 How to read the 3 Panels:
- **Panel A (Clean Landscape):** Continuous deterministic function cross-section, showing the exact bowl, valley, or multimodal peaks and the global basin.
- **Panel B (Homoscedastic Additive Noise, $\sigma = 0.05$):** Additive noise $\mathcal{N}(0, (0.05 \cdot \bar{\Delta y})^2)$ with constant variance across the whole domain, showing persistent evaluation uncertainty at ^*$.
- **Panel C (Heteroscedastic Optimality-Gap Noise, $\sigma = 0.05$):** Gap-dependent noise $\mathcal{N}(0, (0.05 \cdot |f(x) - f^*|)^2)$ with variance vanishing smoothly to **exact zero** at ^*$, preserving basin geometry.


In [8]:
n_pts = 400
t = np.linspace(-5.0, 5.0, n_pts)
n_repeats = 6
t_scatter = np.repeat(t, n_repeats)
FONT_FAMILY = "Inter, -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif"

for p_id in TARGET_PROBLEM_IDS:
    # Domain Noise Strategy Instances via Factory
    strat_clean = NoiseStrategyFactory.create(NoiseModelEnum.NONE)
    strat_homo = NoiseStrategyFactory.create(NoiseModelEnum.HOMOSCEDASTIC_ADDITIVE, noise_std=primary_noise_std)
    strat_hetero = NoiseStrategyFactory.create(NoiseModelEnum.HETEROSCEDASTIC, noise_std=primary_noise_std)

    # Problem Environments with Injected Domain Strategies
    p_clean = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_clean, instance_id=instance_id)
    p_homo = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_homo, instance_id=instance_id)
    p_hetero = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_hetero, instance_id=instance_id)

    f_opt = p_clean.true_optimum
    x_opt = p_clean.optimum_x
    clean_name = BBOBFunction.get_name(p_id).split(" (")[0].replace(" Multi-Modal", "")
    p_class = BBOBFunction.get_class(p_id)
    fn_slug = f"f{p_id}_{p_clean.name.lower()}"
    fn_dir = pub_dir / fn_slug
    fn_dir.mkdir(parents=True, exist_ok=True)

    # 1D slice through optimum along dimension 0
    pts_1d = np.tile(x_opt, (n_pts, 1))
    pts_1d[:, 0] = t
    y_clean = np.array([p_clean(pt) for pt in pts_1d])

    pts_scatter = np.repeat(pts_1d, n_repeats, axis=0)
    y_clean_rep = np.repeat(y_clean, n_repeats)

    # 1. Homoscedastic Additive Noise: Sampled strictly via BBOBProblem(pt) -> noise_strategy.add_noise()
    homo_std_val = strat_homo.noise_std * strat_homo.landscape_scale
    homo_std_curve = np.full_like(y_clean, homo_std_val)
    np.random.seed(42)
    y_homo_scatter = np.array([p_homo(pt) for pt in pts_scatter])

    # 2. Heteroscedastic Optimality-Gap Noise: Sampled strictly via BBOBProblem(pt) -> noise_strategy.add_noise()
    hetero_std_curve = strat_hetero.noise_std * np.abs(y_clean - strat_hetero.true_optimum)
    np.random.seed(42)
    y_hetero_scatter = np.array([p_hetero(pt) for pt in pts_scatter])

    fig_3p = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            f"<b>(A) Clean {clean_name} (f{p_id})</b>",
            f"<b>(B) Homoscedastic Additive Noise (σ={primary_noise_std})</b>",
            f"<b>(C) Heteroscedastic Optimality-Gap Noise (σ={primary_noise_std})</b>"
        ],
        horizontal_spacing=0.08
    )

    # Panel A: Clean
    fig_3p.add_trace(go.Scatter(
        x=t, y=y_clean, mode="lines", name=f"Clean f{p_id}",
        line=dict(color="#0F172A", width=3.2)
    ), row=1, col=1)
    fig_3p.add_trace(go.Scatter(
        x=[x_opt[0]], y=[f_opt], mode="markers",
        name="Global Optimum (x*)",
        marker=dict(color="#EF4444", size=14, symbol="star", line=dict(width=1.8, color="#FFFFFF"))
    ), row=1, col=1)

    # Panel B: Homoscedastic Additive
    fig_3p.add_trace(go.Scatter(
        x=np.concatenate([t, t[::-1]]),
        y=np.concatenate([y_clean + 2 * homo_std_curve, (y_clean - 2 * homo_std_curve)[::-1]]),
        fill="toself", fillcolor="rgba(239, 68, 68, 0.18)",
        line=dict(color="rgba(255,255,255,0)"), name="±2σ Homoscedastic Band"
    ), row=1, col=2)
    fig_3p.add_trace(go.Scatter(
        x=t_scatter, y=y_homo_scatter, mode="markers", name="Homoscedastic Evals",
        marker=dict(color="#EF4444", size=4.5, opacity=0.45)
    ), row=1, col=2)
    fig_3p.add_trace(go.Scatter(
        x=t, y=y_clean, mode="lines", name="Clean Reference",
        line=dict(color="#0F172A", width=1.8, dash="dash")
    ), row=1, col=2)

    # Panel C: Heteroscedastic Optimality Gap
    fig_3p.add_trace(go.Scatter(
        x=np.concatenate([t, t[::-1]]),
        y=np.concatenate([y_clean + 2 * hetero_std_curve, (y_clean - 2 * hetero_std_curve)[::-1]]),
        fill="toself", fillcolor="rgba(16, 185, 129, 0.22)",
        line=dict(color="rgba(255,255,255,0)"), name="±2σ Heteroscedastic Band"
    ), row=1, col=3)
    fig_3p.add_trace(go.Scatter(
        x=t_scatter, y=y_hetero_scatter, mode="markers", name="Heteroscedastic Evals",
        marker=dict(color="#10B981", size=4.5, opacity=0.45)
    ), row=1, col=3)
    fig_3p.add_trace(go.Scatter(
        x=t, y=y_clean, mode="lines", name="Clean Reference",
        line=dict(color="#0F172A", width=1.8, dash="dash"), showlegend=False
    ), row=1, col=3)
    fig_3p.add_trace(go.Scatter(
        x=[x_opt[0]], y=[f_opt], mode="markers", name="Zero-Variance at x*",
        marker=dict(color="#047857", size=14, symbol="star", line=dict(width=1.8, color="#FFFFFF"))
    ), row=1, col=3)

    fig_3p.update_xaxes(title_text="<b>Spatial Coordinate x₁</b> (x₂ = x₂*)", title_font=dict(size=18, family=FONT_FAMILY), tickfont=dict(size=15, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1")
    fig_3p.update_yaxes(title_text="<b>Objective f(x)</b>", title_font=dict(size=18, family=FONT_FAMILY), tickfont=dict(size=15, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", row=1, col=1)
    fig_3p.update_yaxes(tickfont=dict(size=15, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", row=1, col=2)
    fig_3p.update_yaxes(tickfont=dict(size=15, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", row=1, col=3)

    for anno in fig_3p.layout.annotations:
        anno.update(font=dict(size=18, color="#0F172A", family=FONT_FAMILY), yshift=14)

    fig_3p.update_layout(
        title=dict(
            text=f"<b>Figure: 1D Noise Cross-Section — BBOB f{p_id} ({clean_name})</b><br><span style='font-size:15px;color:#475569;font-weight:normal;'>Deterministic Base vs. Homoscedastic Additive Noise vs. Heteroscedastic Optimality-Gap Formulation (σ={primary_noise_std})</span>",
            x=0.02, xanchor="left", y=0.98,
            pad=dict(b=20, t=10),
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY)
        ),
        height=580, width=1380,
        margin=dict(l=80, r=45, t=150, b=110),
        template="plotly_white",
        legend=dict(
            orientation="h",
            y=-0.22, yanchor="top",
            x=0.5, xanchor="center",
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="#E2E8F0",
            borderwidth=1,
            font=dict(size=15, family=FONT_FAMILY)
        )
    )

    out_p = fn_dir / "figure_1d_noise_cross_section.png"
    fig_3p.write_image(str(out_p), scale=3)
    print(f"✅ Exported 1D 3-panel figure: {fn_slug}/figure_1d_noise_cross_section.png")


2026-09-18 23:01:35 INFO Chromium init'ed with kwargs {}
2026-09-18 23:01:35 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-18 23:01:35 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppgnc2imh.
2026-09-18 23:01:35 INFO Opening browser.
2026-09-18 23:01:35 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpfhh3g1yw.
2026-09-18 23:01:35 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpfhh3g1yw
2026-09-18 23:01:37 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppgnc2imh/index.html
2026-09-18 23:01:38 INFO Getting tab from queue (has 1)
2026-09-18 23:01:38 INFO Got 43C1
2026-09-18 23:01:38 INFO Reloading tab 43C1 before return.
2026-09-18 23:01:38 INFO Putting tab 43C1 back (queue size: 0).
2026-09-18 23:01:38 INFO Waiting for all cleanups to finish.
2026-09-18 23:01:38 INFO Exiting Kaleido.
2026-09-18 23:01:38 INFO T

✅ Exported 1D 3-panel figure: f1_sphere/figure_1d_noise_cross_section.png


2026-09-18 23:01:40 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmps_yd9s_5/index.html
2026-09-18 23:01:41 INFO Getting tab from queue (has 1)
2026-09-18 23:01:41 INFO Got 0CB1
2026-09-18 23:01:42 INFO Reloading tab 0CB1 before return.
2026-09-18 23:01:42 INFO Putting tab 0CB1 back (queue size: 0).
2026-09-18 23:01:42 INFO Waiting for all cleanups to finish.
2026-09-18 23:01:42 INFO Exiting Kaleido.
2026-09-18 23:01:42 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:42 INFO shutil.rmtree worked.
2026-09-18 23:01:42 INFO Closing browser.
2026-09-18 23:01:42 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:42 INFO shutil.rmtree worked.
2026-09-18 23:01:42 INFO Closing browser.
2026-09-18 23:01:42 INFO Cancelling tasks.
2026-09-18 23:01:42 INFO Exiting Kaleido/Choreo.
2026-09-18 23:01:42 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:42 INFO shutil.rmtree worked.
2026-09-18 23:01:42 INFO Cancelling tasks.
2026-09-18 23:01:4

✅ Exported 1D 3-panel figure: f8_rosenbrock/figure_1d_noise_cross_section.png


2026-09-18 23:01:43 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5dtxxm4x/index.html
2026-09-18 23:01:44 INFO Getting tab from queue (has 1)
2026-09-18 23:01:44 INFO Got F495
2026-09-18 23:01:44 INFO Reloading tab F495 before return.
2026-09-18 23:01:44 INFO Putting tab F495 back (queue size: 0).
2026-09-18 23:01:44 INFO Waiting for all cleanups to finish.
2026-09-18 23:01:44 INFO Exiting Kaleido.
2026-09-18 23:01:44 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:44 INFO shutil.rmtree worked.
2026-09-18 23:01:44 INFO Closing browser.
2026-09-18 23:01:44 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:44 INFO shutil.rmtree worked.
2026-09-18 23:01:44 INFO Closing browser.
2026-09-18 23:01:44 INFO Cancelling tasks.
2026-09-18 23:01:44 INFO Exiting Kaleido/Choreo.
2026-09-18 23:01:44 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:44 INFO shutil.rmtree worked.
2026-09-18 23:01:44 INFO Cancelling tasks.
2026-09-18 23:01:4

✅ Exported 1D 3-panel figure: f11_discus/figure_1d_noise_cross_section.png


2026-09-18 23:01:45 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmptx8ucuha/index.html
2026-09-18 23:01:46 INFO Getting tab from queue (has 1)
2026-09-18 23:01:46 INFO Got A03E
2026-09-18 23:01:47 INFO Reloading tab A03E before return.
2026-09-18 23:01:47 INFO Putting tab A03E back (queue size: 0).
2026-09-18 23:01:47 INFO Waiting for all cleanups to finish.
2026-09-18 23:01:47 INFO Exiting Kaleido.
2026-09-18 23:01:47 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:47 INFO shutil.rmtree worked.
2026-09-18 23:01:47 INFO Closing browser.
2026-09-18 23:01:47 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:47 INFO shutil.rmtree worked.
2026-09-18 23:01:47 INFO Closing browser.
2026-09-18 23:01:47 INFO Cancelling tasks.
2026-09-18 23:01:47 INFO Exiting Kaleido/Choreo.
2026-09-18 23:01:47 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:47 INFO shutil.rmtree worked.
2026-09-18 23:01:47 INFO Cancelling tasks.
2026-09-18 23:01:4

✅ Exported 1D 3-panel figure: f15_rastriginrotated/figure_1d_noise_cross_section.png


2026-09-18 23:01:48 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp0a3zxuix/index.html
2026-09-18 23:01:50 INFO Getting tab from queue (has 1)
2026-09-18 23:01:50 INFO Got 5289
2026-09-18 23:01:50 INFO Reloading tab 5289 before return.
2026-09-18 23:01:50 INFO Putting tab 5289 back (queue size: 0).
2026-09-18 23:01:50 INFO Waiting for all cleanups to finish.
2026-09-18 23:01:50 INFO Exiting Kaleido.
2026-09-18 23:01:50 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:50 INFO shutil.rmtree worked.
2026-09-18 23:01:50 INFO Closing browser.
2026-09-18 23:01:50 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:50 INFO shutil.rmtree worked.
2026-09-18 23:01:50 INFO Closing browser.
2026-09-18 23:01:50 INFO Cancelling tasks.
2026-09-18 23:01:50 INFO Exiting Kaleido/Choreo.
2026-09-18 23:01:51 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:51 INFO shutil.rmtree worked.
2026-09-18 23:01:51 INFO Cancelling tasks.
2026-09-18 23:01:5

✅ Exported 1D 3-panel figure: f21_gallagher101/figure_1d_noise_cross_section.png


## 3. 2D Contour & Basin Topology Comparison Across Functions

### 🎯 Why we do this plot:
To observe how 2D contour level sets and local basins distort across the 2D spatial search domain under the three noise regimes (Clean vs. Homoscedastic Additive vs. Heteroscedastic Optimality-Gap) for every function class.


In [9]:
grid_resol = 70
x_range = np.linspace(-5.0, 5.0, grid_resol)
y_range = np.linspace(-5.0, 5.0, grid_resol)
X, Y = np.meshgrid(x_range, y_range)
FONT_FAMILY = "Inter, -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif"

def to_rank_norm(Z):
    zf = Z.flatten()
    zr = rankdata(zf)
    return ((zr - zr.min()) / (zr.max() - zr.min())).reshape(Z.shape)

for p_id in TARGET_PROBLEM_IDS:
    # Domain Noise Strategy Instances via Factory
    strat_clean = NoiseStrategyFactory.create(NoiseModelEnum.NONE)
    strat_homo = NoiseStrategyFactory.create(NoiseModelEnum.HOMOSCEDASTIC_ADDITIVE, noise_std=primary_noise_std)
    strat_hetero = NoiseStrategyFactory.create(NoiseModelEnum.HETEROSCEDASTIC, noise_std=primary_noise_std)

    # Problem Environments with Injected Domain Strategies
    p_clean = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_clean, instance_id=instance_id)
    p_homo = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_homo, instance_id=instance_id)
    p_hetero = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_hetero, instance_id=instance_id)

    f_opt = p_clean.true_optimum
    x_opt = p_clean.optimum_x
    clean_name = BBOBFunction.get_name(p_id).split(" (")[0].replace(" Multi-Modal", "")
    p_class = BBOBFunction.get_class(p_id)
    fn_slug = f"f{p_id}_{p_clean.name.lower()}"
    fn_dir = pub_dir / fn_slug
    fn_dir.mkdir(parents=True, exist_ok=True)

    # Evaluate Grids strictly through BBOBProblem.__call__ -> noise_strategy.add_noise()
    Z_clean = np.zeros_like(X)
    for i in range(grid_resol):
        for j in range(grid_resol):
            Z_clean[i, j] = p_clean([X[i, j], Y[i, j]])

    np.random.seed(42)
    Z_homo = np.zeros_like(X)
    for i in range(grid_resol):
        for j in range(grid_resol):
            Z_homo[i, j] = p_homo([X[i, j], Y[i, j]])

    np.random.seed(42)
    Z_hetero = np.zeros_like(X)
    for i in range(grid_resol):
        for j in range(grid_resol):
            Z_hetero[i, j] = p_hetero([X[i, j], Y[i, j]])

    fig_contour_3p = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            f"<b>(A) Clean {clean_name} (f{p_id})</b>",
            f"<b>(B) Homoscedastic Additive Noise (σ={primary_noise_std})</b>",
            f"<b>(C) Heteroscedastic Optimality-Gap Noise (σ={primary_noise_std})</b>"
        ],
        horizontal_spacing=0.08
    )

    grids = [("Clean", to_rank_norm(Z_clean)), ("Homoscedastic", to_rank_norm(Z_homo)), ("Heteroscedastic", to_rank_norm(Z_hetero))]
    for idx, (label, z_grid) in enumerate(grids, start=1):
        fig_contour_3p.add_trace(
            go.Contour(
                x=x_range, y=y_range, z=z_grid,
                colorscale="Viridis",
                showscale=False,
                colorbar=dict(
                    title=dict(text="<b>Rank Norm</b>", font=dict(size=16, family=FONT_FAMILY)),
                    tickfont=dict(size=14, family=FONT_FAMILY),
                    thickness=16, len=0.85, y=0.5, yanchor="middle"
                ) if idx == 3 else None,
                contours=dict(coloring="heatmap", showlines=True)
            ),
            row=1, col=idx
        )
        fig_contour_3p.add_trace(
            go.Scatter(
                x=[x_opt[0]], y=[x_opt[1]], mode="markers",
                marker=dict(color="#FFFFFF", symbol="x", size=13, line=dict(width=2.8, color="#EF4444")),
                showlegend=False, name="Global Optimum x*"
            ),
            row=1, col=idx
        )

    for anno in fig_contour_3p.layout.annotations:
        anno.update(font=dict(size=18, color="#0F172A", family=FONT_FAMILY), yshift=14)

    fig_contour_3p.update_xaxes(title_text="<b>Coordinate x₁</b>", title_font=dict(size=18, family=FONT_FAMILY), tickfont=dict(size=15, family=FONT_FAMILY), showgrid=True, gridcolor="#E2E8F0", linecolor="#94A3B8")
    fig_contour_3p.update_yaxes(title_text="<b>Coordinate x₂</b>", title_font=dict(size=18, family=FONT_FAMILY), tickfont=dict(size=15, family=FONT_FAMILY), showgrid=True, gridcolor="#E2E8F0", linecolor="#94A3B8")

    fig_contour_3p.update_layout(
        title=dict(
            text=f"<b>Figure: 2D Contour Topology Distortion — BBOB f{p_id} ({clean_name})</b><br><span style='font-size:15px;color:#475569;font-weight:normal;'>Rank-Normalized Iso-Fitness Contours under Clean vs. Homoscedastic Additive vs. Heteroscedastic Optimality-Gap Regimes (σ={primary_noise_std})</span>",
            x=0.02, xanchor="left", y=0.98,
            pad=dict(b=20, t=10),
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY)
        ),
        height=540, width=1380,
        margin=dict(l=75, r=75, t=150, b=65),
        template="plotly_white"
    )

    out_p = fn_dir / "figure_2d_noise_topology.png"
    fig_contour_3p.write_image(str(out_p), scale=3)
    print(f"✅ Exported 2D 3-panel figure: {fn_slug}/figure_2d_noise_topology.png")


2026-09-18 23:01:51 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:51 INFO shutil.rmtree worked.
2026-09-18 23:01:51 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:51 INFO shutil.rmtree worked.
2026-09-18 23:01:51 INFO Chromium init'ed with kwargs {}
2026-09-18 23:01:51 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-18 23:01:51 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpj3oz671j.
2026-09-18 23:01:51 INFO Opening browser.
2026-09-18 23:01:51 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp4bijipf_.
2026-09-18 23:01:51 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp4bijipf_
2026-09-18 23:01:52 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpj3oz671j/index.html
2026-09-18 23:01:53 INFO Getting tab from queue (has 1)
2026-09-18 23:01:53 INFO Got E71D
2026-09-18 23:01:53 INFO Reloading

✅ Exported 2D 3-panel figure: f1_sphere/figure_2d_noise_topology.png


2026-09-18 23:01:54 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpxxog3ont/index.html
2026-09-18 23:01:55 INFO Getting tab from queue (has 1)
2026-09-18 23:01:55 INFO Got 1CBD
2026-09-18 23:01:56 INFO Reloading tab 1CBD before return.
2026-09-18 23:01:56 INFO Putting tab 1CBD back (queue size: 0).
2026-09-18 23:01:56 INFO Waiting for all cleanups to finish.
2026-09-18 23:01:56 INFO Exiting Kaleido.
2026-09-18 23:01:56 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:56 INFO shutil.rmtree worked.
2026-09-18 23:01:56 INFO Closing browser.
2026-09-18 23:01:56 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:56 INFO shutil.rmtree worked.
2026-09-18 23:01:56 INFO Closing browser.
2026-09-18 23:01:56 INFO Cancelling tasks.
2026-09-18 23:01:56 INFO Exiting Kaleido/Choreo.
2026-09-18 23:01:56 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:56 INFO shutil.rmtree worked.
2026-09-18 23:01:56 INFO Cancelling tasks.
2026-09-18 23:01:5

✅ Exported 2D 3-panel figure: f8_rosenbrock/figure_2d_noise_topology.png


2026-09-18 23:01:56 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpz6mo3__1/index.html
2026-09-18 23:01:57 INFO Getting tab from queue (has 1)
2026-09-18 23:01:57 INFO Got E5DC
2026-09-18 23:01:57 INFO Reloading tab E5DC before return.
2026-09-18 23:01:57 INFO Putting tab E5DC back (queue size: 0).
2026-09-18 23:01:57 INFO Waiting for all cleanups to finish.
2026-09-18 23:01:57 INFO Exiting Kaleido.
2026-09-18 23:01:57 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:57 INFO shutil.rmtree worked.
2026-09-18 23:01:57 INFO Closing browser.
2026-09-18 23:01:57 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:57 INFO shutil.rmtree worked.
2026-09-18 23:01:57 INFO Closing browser.
2026-09-18 23:01:57 INFO Cancelling tasks.
2026-09-18 23:01:57 INFO Exiting Kaleido/Choreo.
2026-09-18 23:01:58 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:01:58 INFO shutil.rmtree worked.
2026-09-18 23:01:58 INFO Cancelling tasks.
2026-09-18 23:01:5

✅ Exported 2D 3-panel figure: f11_discus/figure_2d_noise_topology.png


2026-09-18 23:01:58 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpoh7_yth3/index.html
2026-09-18 23:01:59 INFO Getting tab from queue (has 1)
2026-09-18 23:01:59 INFO Got 8688
2026-09-18 23:02:00 INFO Reloading tab 8688 before return.
2026-09-18 23:02:00 INFO Putting tab 8688 back (queue size: 0).
2026-09-18 23:02:00 INFO Waiting for all cleanups to finish.
2026-09-18 23:02:00 INFO Exiting Kaleido.
2026-09-18 23:02:00 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:00 INFO shutil.rmtree worked.
2026-09-18 23:02:00 INFO Closing browser.
2026-09-18 23:02:00 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:00 INFO shutil.rmtree worked.
2026-09-18 23:02:00 INFO Closing browser.
2026-09-18 23:02:00 INFO Cancelling tasks.
2026-09-18 23:02:00 INFO Exiting Kaleido/Choreo.
2026-09-18 23:02:00 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:00 INFO shutil.rmtree worked.
2026-09-18 23:02:00 INFO Cancelling tasks.
2026-09-18 23:02:0

✅ Exported 2D 3-panel figure: f15_rastriginrotated/figure_2d_noise_topology.png


2026-09-18 23:02:00 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpxs5bsio8/index.html
2026-09-18 23:02:01 INFO Getting tab from queue (has 1)
2026-09-18 23:02:01 INFO Got E3C8
2026-09-18 23:02:02 INFO Reloading tab E3C8 before return.
2026-09-18 23:02:03 INFO Putting tab E3C8 back (queue size: 0).
2026-09-18 23:02:03 INFO Waiting for all cleanups to finish.
2026-09-18 23:02:03 INFO Exiting Kaleido.
2026-09-18 23:02:03 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:03 INFO shutil.rmtree worked.
2026-09-18 23:02:03 INFO Closing browser.
2026-09-18 23:02:03 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:03 INFO shutil.rmtree worked.
2026-09-18 23:02:03 INFO Closing browser.
2026-09-18 23:02:03 INFO Cancelling tasks.
2026-09-18 23:02:03 INFO Exiting Kaleido/Choreo.
2026-09-18 23:02:03 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:03 INFO shutil.rmtree worked.
2026-09-18 23:02:03 INFO Cancelling tasks.
2026-09-18 23:02:0

✅ Exported 2D 3-panel figure: f21_gallagher101/figure_2d_noise_topology.png


## 4. 3D Surface Landscape Topology Comparison Across Functions

### 🏔️ 3D Perspective Visualizations
We render 3-panel 3D surface plots for all target benchmark functions (, f_8, f_{11}, f_{15}, f_{21}$):
- **Panel A (Clean 3D Surface):** Unperturbed mathematical topology showing true hills, valleys, and global optimum basin.
- **Panel B (Homoscedastic Additive Noise):** Uniform additive noise fluctuations across the entire surface.
- **Panel C (Heteroscedastic Optimality-Gap Noise):** State-dependent boundary noise gradients with a smooth, preserved central basin around ^*$.

Exports high-resolution figures to .


In [10]:
grid_resol_3d = 50
x_range_3d = np.linspace(-5.0, 5.0, grid_resol_3d)
y_range_3d = np.linspace(-5.0, 5.0, grid_resol_3d)
X_3d, Y_3d = np.meshgrid(x_range_3d, y_range_3d)
FONT_FAMILY = "Inter, -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif"

camera_view = dict(
    eye=dict(x=1.45, y=-1.45, z=1.25),
    center=dict(x=0, y=0, z=-0.12)
)

for p_id in TARGET_PROBLEM_IDS:
    # Domain Noise Strategy Instances via Factory
    strat_clean = NoiseStrategyFactory.create(NoiseModelEnum.NONE)
    strat_homo = NoiseStrategyFactory.create(NoiseModelEnum.HOMOSCEDASTIC_ADDITIVE, noise_std=primary_noise_std)
    strat_hetero = NoiseStrategyFactory.create(NoiseModelEnum.HETEROSCEDASTIC, noise_std=primary_noise_std)

    # Problem Environments with Injected Domain Strategies
    p_clean = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_clean, instance_id=instance_id)
    p_homo = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_homo, instance_id=instance_id)
    p_hetero = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=strat_hetero, instance_id=instance_id)

    f_opt = p_clean.true_optimum
    x_opt = p_clean.optimum_x
    clean_name = BBOBFunction.get_name(p_id).split(" (")[0].replace(" Multi-Modal", "")
    p_class = BBOBFunction.get_class(p_id)
    fn_slug = f"f{p_id}_{p_clean.name.lower()}"
    fn_dir = pub_dir / fn_slug
    fn_dir.mkdir(parents=True, exist_ok=True)

    # Evaluate Grids strictly through BBOBProblem.__call__ -> noise_strategy.add_noise()
    Z_clean = np.zeros_like(X_3d)
    for i in range(grid_resol_3d):
        for j in range(grid_resol_3d):
            Z_clean[i, j] = p_clean([X_3d[i, j], Y_3d[i, j]])

    np.random.seed(42)
    Z_homo = np.zeros_like(X_3d)
    for i in range(grid_resol_3d):
        for j in range(grid_resol_3d):
            Z_homo[i, j] = p_homo([X_3d[i, j], Y_3d[i, j]])

    np.random.seed(42)
    Z_hetero = np.zeros_like(X_3d)
    for i in range(grid_resol_3d):
        for j in range(grid_resol_3d):
            Z_hetero[i, j] = p_hetero([X_3d[i, j], Y_3d[i, j]])

    fig_3d = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "surface"}, {"type": "surface"}, {"type": "surface"}]],
        subplot_titles=[
            f"<b>(A) Clean Surface (f{p_id})</b>",
            f"<b>(B) Homoscedastic Additive Noise (σ={primary_noise_std})</b>",
            f"<b>(C) Heteroscedastic Optimality-Gap Noise (σ={primary_noise_std})</b>"
        ],
        horizontal_spacing=0.04
    )

    # Clean Surface
    fig_3d.add_trace(go.Surface(x=X_3d, y=Y_3d, z=Z_clean, colorscale="Viridis", showscale=False, opacity=0.96), row=1, col=1)
    # Homoscedastic Surface
    fig_3d.add_trace(go.Surface(x=X_3d, y=Y_3d, z=Z_homo, colorscale="Inferno", showscale=False, opacity=0.92), row=1, col=2)
    # Heteroscedastic Surface
    fig_3d.add_trace(go.Surface(x=X_3d, y=Y_3d, z=Z_hetero, colorscale="Viridis", showscale=False, opacity=0.96), row=1, col=3)

    scene_style = dict(
        camera=camera_view,
        xaxis=dict(title=dict(text="<b>x₁</b>", font=dict(size=16, family=FONT_FAMILY)), tickfont=dict(size=13, family=FONT_FAMILY), backgroundcolor="#F8FAFC", gridcolor="#E2E8F0"),
        yaxis=dict(title=dict(text="<b>x₂</b>", font=dict(size=16, family=FONT_FAMILY)), tickfont=dict(size=13, family=FONT_FAMILY), backgroundcolor="#F8FAFC", gridcolor="#E2E8F0"),
        zaxis=dict(title=dict(text="<b>f(x)</b>", font=dict(size=16, family=FONT_FAMILY)), tickfont=dict(size=13, family=FONT_FAMILY), backgroundcolor="#F8FAFC", gridcolor="#E2E8F0"),
    )

    for anno in fig_3d.layout.annotations:
        anno.update(font=dict(size=18, color="#0F172A", family=FONT_FAMILY), yshift=14)

    fig_3d.update_layout(
        title=dict(
            text=f"<b>Figure: 3D Fitness Landscape Topology — BBOB f{p_id} ({clean_name})</b><br><span style='font-size:15px;color:#475569;font-weight:normal;'>3D Perspective Comparison of Clean vs. Homoscedastic Additive vs. Heteroscedastic Optimality-Gap Noise (σ={primary_noise_std})</span>",
            x=0.02, xanchor="left", y=0.98,
            pad=dict(b=20, t=10),
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY)
        ),
        scene=scene_style,
        scene2=scene_style,
        scene3=scene_style,
        height=600,
        width=1480,
        margin=dict(l=40, r=40, t=150, b=45),
        template="plotly_white"
    )

    out_p = fn_dir / "figure_3d_noise_surface.png"
    fig_3d.write_image(str(out_p), scale=3)
    print(f"✅ Exported 3D 3-panel surface figure: {fn_slug}/{out_p.name}")


2026-09-18 23:02:03 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:03 INFO shutil.rmtree worked.
2026-09-18 23:02:03 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:03 INFO shutil.rmtree worked.
2026-09-18 23:02:03 INFO Chromium init'ed with kwargs {}
2026-09-18 23:02:03 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-18 23:02:03 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp9akj4t7c.
2026-09-18 23:02:03 INFO Opening browser.
2026-09-18 23:02:03 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpc6mx5prz.
2026-09-18 23:02:03 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpc6mx5prz
2026-09-18 23:02:04 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp9akj4t7c/index.html
2026-09-18 23:02:05 INFO Getting tab from queue (has 1)
2026-09-18 23:02:05 INFO Got 112C
2026-09-18 23:02:11 INFO Reloading

✅ Exported 3D 3-panel surface figure: f1_sphere/figure_3d_noise_surface.png


2026-09-18 23:02:14 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpletg93ht/index.html
2026-09-18 23:02:14 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:14 INFO shutil.rmtree worked.
2026-09-18 23:02:14 INFO Getting tab from queue (has 1)
2026-09-18 23:02:14 INFO Got B44D
2026-09-18 23:02:20 INFO Reloading tab B44D before return.
2026-09-18 23:02:20 INFO Putting tab B44D back (queue size: 0).
2026-09-18 23:02:20 INFO Waiting for all cleanups to finish.
2026-09-18 23:02:20 INFO Exiting Kaleido.
2026-09-18 23:02:20 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:20 INFO shutil.rmtree worked.
2026-09-18 23:02:20 INFO Closing browser.
2026-09-18 23:02:20 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:20 INFO shutil.rmtree worked.
2026-09-18 23:02:20 INFO Closing browser.
2026-09-18 23:02:20 INFO Cancelling tasks.
2026-09-18 23:02:20 INFO Exiting Kaleido/Choreo.
2026-09-18 23:02:20 INFO TemporaryDirectory.cleanup() worked.

✅ Exported 3D 3-panel surface figure: f8_rosenbrock/figure_3d_noise_surface.png


2026-09-18 23:02:21 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmps0v2sqam/index.html
2026-09-18 23:02:22 INFO Getting tab from queue (has 1)
2026-09-18 23:02:22 INFO Got 950E
2026-09-18 23:02:28 INFO Reloading tab 950E before return.
2026-09-18 23:02:28 INFO Putting tab 950E back (queue size: 0).
2026-09-18 23:02:28 INFO Waiting for all cleanups to finish.
2026-09-18 23:02:28 INFO Exiting Kaleido.
2026-09-18 23:02:28 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:28 INFO shutil.rmtree worked.
2026-09-18 23:02:28 INFO Closing browser.
2026-09-18 23:02:28 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:28 INFO shutil.rmtree worked.
2026-09-18 23:02:28 INFO Closing browser.
2026-09-18 23:02:28 INFO Cancelling tasks.
2026-09-18 23:02:28 INFO Exiting Kaleido/Choreo.
2026-09-18 23:02:28 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:28 INFO shutil.rmtree worked.
2026-09-18 23:02:28 INFO Cancelling tasks.
2026-09-18 23:02:2

✅ Exported 3D 3-panel surface figure: f11_discus/figure_3d_noise_surface.png


2026-09-18 23:02:29 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppwfrutuu/index.html
2026-09-18 23:02:30 INFO Getting tab from queue (has 1)
2026-09-18 23:02:30 INFO Got A699
2026-09-18 23:02:37 INFO Reloading tab A699 before return.
2026-09-18 23:02:37 INFO Putting tab A699 back (queue size: 0).
2026-09-18 23:02:37 INFO Waiting for all cleanups to finish.
2026-09-18 23:02:37 INFO Exiting Kaleido.
2026-09-18 23:02:37 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:37 INFO shutil.rmtree worked.
2026-09-18 23:02:37 INFO Closing browser.
2026-09-18 23:02:37 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:37 INFO shutil.rmtree worked.
2026-09-18 23:02:37 INFO Closing browser.
2026-09-18 23:02:37 INFO Cancelling tasks.
2026-09-18 23:02:37 INFO Exiting Kaleido/Choreo.
2026-09-18 23:02:37 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:37 INFO shutil.rmtree worked.
2026-09-18 23:02:37 INFO Cancelling tasks.
2026-09-18 23:02:3

✅ Exported 3D 3-panel surface figure: f15_rastriginrotated/figure_3d_noise_surface.png


2026-09-18 23:02:38 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpdlgsk46v/index.html
2026-09-18 23:02:40 INFO Getting tab from queue (has 1)
2026-09-18 23:02:40 INFO Got 7F1C
2026-09-18 23:02:46 INFO Reloading tab 7F1C before return.
2026-09-18 23:02:46 INFO Putting tab 7F1C back (queue size: 0).
2026-09-18 23:02:46 INFO Waiting for all cleanups to finish.
2026-09-18 23:02:46 INFO Exiting Kaleido.
2026-09-18 23:02:46 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:46 INFO shutil.rmtree worked.
2026-09-18 23:02:46 INFO Closing browser.
2026-09-18 23:02:46 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:46 INFO shutil.rmtree worked.
2026-09-18 23:02:46 INFO Closing browser.
2026-09-18 23:02:46 INFO Cancelling tasks.
2026-09-18 23:02:46 INFO Exiting Kaleido/Choreo.
2026-09-18 23:02:46 INFO TemporaryDirectory.cleanup() worked.
2026-09-18 23:02:46 INFO shutil.rmtree worked.
2026-09-18 23:02:46 INFO Cancelling tasks.
2026-09-18 23:02:4

✅ Exported 3D 3-panel surface figure: f21_gallagher101/figure_3d_noise_surface.png
